# Scorer fit scope: `unseen` vs `all`

The anomaly scorer learns its *normal residual* baseline from a set of rows, then
flags evaluation rows whose residual is unusual relative to that baseline. **Which
rows the scorer is allowed to learn from** is controlled by `detect.scorer_fit_scope`:

| scope | scorer fits on | residuals are | bias |
|-------|----------------|---------------|------|
| `"unseen"` (default) | only the held-out **test** split (data unseen by the forecaster trainer *and* the tuner) | genuine out-of-sample | none — honest baseline |
| `"all"` | the **entire** dataset (train + val + test) | in-sample over training rows | optimistic — the model already fit those rows, so residuals are artificially small |

Because the forecaster was *fit* on the training rows, its residuals there are
unusually small. Feeding those small residuals into the scorer shrinks the learned
"normal" spread, which raises the detection threshold's effective strictness on the
eval window and can **mask** real anomalies. The `"unseen"` scope avoids this by
calibrating only on data the pipeline never used for fitting or selection.

This notebook runs detection **both ways on the same data and model** and compares
the resulting scores and flags so the effect is measurable rather than theoretical.

In [ ]:
from __future__ import annotations

import copy
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd

from spotanomaly2.application.config import load_config
from spotanomaly2.application.data_manager import DataManager
from spotanomaly2.domain.anomaly_detector import AnomalyDetector
from spotanomaly2.infrastructure import logging

# Point this at your runtime config (falls back to the bundled default).
CONFIG_PATH = Path("../config/config.yaml")
if not CONFIG_PATH.exists():
    CONFIG_PATH = Path("../config/default.yaml")

logger = logging.get_logger("compare_scorer_fit_scope")
config = load_config(CONFIG_PATH, base_dir=CONFIG_PATH.resolve().parent.parent)
print(f"Loaded config from {CONFIG_PATH}")

In [ ]:
# Load the processed panels. Detection needs both processed data and a trained
# model on disk — run `uv run spotanomaly2 all` (or process + train) first if this
# raises.
data_manager = DataManager(config, logger)
panel_data = data_manager.load_processed_data()
print(f"Loaded {len(panel_data)} panel(s): {list(panel_data)}")
for pid, df in panel_data.items():
    print(f"  panel {pid}: {len(df)} rows, {df.index.min()} → {df.index.max()}")

In [ ]:
# --- Train fresh models with a large held-out window -------------------------
# Both scopes reuse whatever model is on disk, so to compare them on a *big*
# evaluation window we retrain here and pin the resulting model for the runs
# below. Training just fits one forecaster per channel with the resolved
# per-channel params (no hyperparameter search), so this is reasonably quick.
#
# Which window each scope calibrates on:
#   scorer_fit_scope="unseen" -> the **test** split (held out from fit AND tuning)
#   scorer_fit_scope="all"    -> the entire series
# A big `test` split is what gives the "unseen" scorer a large, leakage-free
# window; `train` must still be large enough to fit a decent forecaster. With
# detect.hist_window=8640 the test split needs >= 2*8640+1 rows, otherwise the
# detector reshapes "unseen" to a shorter insufficient-data window (see
# _adjust_window_for_insufficient_data) and the two scopes stop being comparable.
from spotanomaly2.domain.model_trainer import ModelTrainer

BIG_SPLIT = {"train": 50, "val": 10, "test": 40}  # bigger `test` -> bigger unseen window

train_config = copy.deepcopy(config)
train_config["train"]["split"] = BIG_SPLIT
# This config.yaml predates the train.model -> train.fallback_model rename; the
# trainer reads fallback_model, so bridge the old key if the new one is absent.
train_config["train"].setdefault("fallback_model", train_config["train"].get("model", "LightGBM"))

trainer = ModelTrainer(train_config, logger)
train_results = trainer.run(panel_data)
TRAINED_TIMESTAMP = next(iter(train_results.values()))[1]
print(f"Trained fresh models (timestamp={TRAINED_TIMESTAMP}) with split={BIG_SPLIT}")
for pid, (eval_df, _) in train_results.items():
    print(f"  panel {pid}: {len(eval_df)} channel(s) trained")


In [ ]:
def run_with_scope(scope: str) -> dict[str, tuple]:
    """Run detection over all panels with a given ``detect.scorer_fit_scope``.

    Deep-copies the config so the two runs are fully independent and pins the
    freshly trained model (TRAINED_TIMESTAMP) so the same model artifact /
    processed data is reused for both — the only thing that changes is which
    rows the scorer learns from.
    """
    cfg = copy.deepcopy(config)
    cfg["detect"]["scorer_fit_scope"] = scope
    cfg["detect"]["model_timestamp"] = TRAINED_TIMESTAMP  # pin the model trained above
    detector = AnomalyDetector(cfg, logger)
    return detector.run(panel_data)


results_unseen = run_with_scope("unseen")
results_all = run_with_scope("all")
print("Done. Both strategies evaluated on the same data + freshly trained model.")


In [ ]:
def summarize(panel_id: str) -> dict:
    """Per-panel comparison of the two strategies on the shared eval window.

    The eval window can differ slightly between runs (scope='all' makes the whole
    series 'unseen', so the tail eval window is identical, but leading invalid
    rows may be dropped differently). We align on the shared timestamps before
    comparing flags so the overlap metric is apples-to-apples.
    """
    scores_u, flags_u = results_unseen[panel_id][0], results_unseen[panel_id][1]
    scores_a, flags_a = results_all[panel_id][0], results_all[panel_id][1]

    common = flags_u.index.intersection(flags_a.index)
    fu = flags_u.loc[common, "anomaly_flag"].astype(bool)
    fa = flags_a.loc[common, "anomaly_flag"].astype(bool)

    union = int((fu | fa).sum())
    overlap = int((fu & fa).sum())
    jaccard = overlap / union if union else float("nan")

    return {
        "panel": panel_id,
        "eval_rows": len(common),
        "flags_unseen": int(fu.sum()),
        "flags_all": int(fa.sum()),
        "rate_unseen_%": round(100 * fu.mean(), 3),
        "rate_all_%": round(100 * fa.mean(), 3),
        "only_unseen": int((fu & ~fa).sum()),
        "only_all": int((fa & ~fu).sum()),
        "both": overlap,
        "jaccard": round(jaccard, 3),
        "score_med_unseen": round(float(scores_u.loc[common, "anomaly_score"].median()), 4),
        "score_med_all": round(float(scores_a.loc[common, "anomaly_score"].median()), 4),
    }


summary = pd.DataFrame([summarize(pid) for pid in panel_data]).set_index("panel")
summary

## Reading the table

- **`flags_*` / `rate_*`** — how many eval timestamps each strategy flags. If `flags_all`
  is consistently **lower** than `flags_unseen`, the optimistic in-sample baseline is
  masking anomalies (the expected failure mode).
- **`only_unseen`** — anomalies the honest baseline catches that the optimistic one
  misses. These are the points most at risk of being silently dropped if you switch
  to `"all"` in production.
- **`only_all`** — extra flags unique to `"all"` (rare; usually noise from the
  distorted baseline).
- **`jaccard`** — agreement between the two flag sets (1.0 = identical).
- **`score_med_*`** — median raw anomaly score; differences trace back to the
  different normal-residual spread each strategy learned.

In [ ]:
# Visual comparison over the eval window: a grid of (panel x scorer scope).
# Each subplot shows that scope's anomaly score over the panel's shared eval
# window with its flagged points marked. Interactive: drag a region to zoom,
# double-click to reset, use the mode-bar (top-right) for pan / box-zoom.
scopes = {"unseen (honest)": results_unseen, "all (optimistic)": results_all}
colors = {"unseen (honest)": "#1f77b4", "all (optimistic)": "#ff7f0e"}
panels = list(panel_data)

titles = [f"Panel {pid} — {label}" for pid in panels for label in scopes]
fig = make_subplots(
    rows=len(panels),
    cols=len(scopes),
    subplot_titles=titles,
    horizontal_spacing=0.07,
    vertical_spacing=min(0.12, 0.6 / max(len(panels), 1)),
)

for r, pid in enumerate(panels, start=1):
    # Shared eval window across scopes for this panel (apples-to-apples x-axis).
    common = results_unseen[pid][0].index
    for res in scopes.values():
        common = common.intersection(res[pid][0].index)

    for c, (label, res) in enumerate(scopes.items(), start=1):
        scores = res[pid][0].loc[common, "anomaly_score"]
        flags = res[pid][1].loc[common, "anomaly_flag"].astype(bool)
        fig.add_trace(
            go.Scatter(
                x=common, y=scores, mode="lines",
                line=dict(color=colors[label], width=1.1),
                name="anomaly_score", legendgroup="score",
                showlegend=(r == 1 and c == 1),
                hovertemplate="%{x}<br>score=%{y:.3f}<extra></extra>",
            ),
            row=r, col=c,
        )
        if flags.any():
            fig.add_trace(
                go.Scatter(
                    x=common[flags], y=scores[flags], mode="markers",
                    marker=dict(color=colors[label], size=7, line=dict(color="white", width=0.5)),
                    name=f"flag ({int(flags.sum())})", legendgroup=label, showlegend=False,
                    hovertemplate="%{x}<br>FLAG score=%{y:.3f}<extra></extra>",
                ),
                row=r, col=c,
            )
        fig.update_yaxes(title_text="anomaly_score", row=r, col=c)
        fig.update_xaxes(title_text="timestamp", row=r, col=c)

fig.update_layout(
    height=360 * len(panels),
    width=580 * len(scopes),
    title_text="Anomaly score over the eval window — unseen vs all",
    template="plotly_white",
)
fig.show()


## Per-group scores and flags

The combined scorer above collapses all channels into one `anomaly_score`. When
`detect.groups` is configured, the pipeline instead scores **each channel group
independently** (Bonferroni-corrected quantile per group, flags OR-ed into the
system flag) — this is the successor to the old per-channel detector and is
often what you want for diagnosis: it tells you *which* group drove a flag, not
just *that* something fired.

The per-group detail is exposed as additive columns on the objects `detect.run`
already returns — no extra call needed:

- `scores_df` (slot `[0]`) carries `group_<name>_raw`, `group_<name>_normalized`
  and `group_<name>_threshold` per group;
- `flags_df` (slot `[1]`) carries `group_<name>_flag` per group.

Because the per-group threshold is fit on the scorer-fit window, which depends on
`scorer_fit_scope`, the per-group flags also differ between `"unseen"` and
`"all"` — the same masking effect shows up here group by group.

> **Note:** if no groups are configured for a panel, scoring runs as a single
> `"system"` scorer and there is no per-group breakdown — the cells below detect
> this and print a message instead of failing. Set `detect.groups` in your
> config to enable grouped scoring.</cell id="16219537">

In [ ]:
# --- Per-group flag summary --------------------------------------------------
# When detect.groups is configured, _score_grouped scores each channel group
# with its own Bonferroni-corrected quantile and OR-s the per-group flags into
# the system flag. The per-group detail is attached as additive columns on the
# objects detect.run already returns (a 4-tuple per panel:
# (scores, flags, report_pred, contributions)):
#   scores[f"group_{name}_raw"]       — that group's raw anomaly score
#   scores[f"group_{name}_threshold"] — train-fitted raw flag threshold
#   scores[f"group_{name}_normalized"]
#   flags[f"group_{name}_flag"]       — 1 where that group flagged
# Because the threshold is fit on the scorer-fit window, it differs by scope.
GROUP_SCOPE = {"unseen (honest)": results_unseen, "all (optimistic)": results_all}
group_colors = {"unseen (honest)": "#1f77b4", "all (optimistic)": "#ff7f0e"}


def group_names(scores_df: pd.DataFrame) -> list[str]:
    """Group labels present in a scores frame (empty when no groups configured)."""
    suffix = "_raw"
    return [c[len("group_") : -len(suffix)] for c in scores_df.columns if c.startswith("group_") and c.endswith(suffix)]


def per_group_rows(pid: str) -> list[dict]:
    rows = []
    for label, res in GROUP_SCOPE.items():
        scores_df, flags_df = res[pid][0], res[pid][1]
        for name in group_names(scores_df):
            flags = flags_df[f"group_{name}_flag"].astype(bool)
            thr = scores_df[f"group_{name}_threshold"].dropna()
            rows.append(
                {
                    "panel": pid,
                    "scope": label,
                    "group": name,
                    "threshold": round(float(thr.iloc[0]), 5) if len(thr) else float("nan"),
                    "flags": int(flags.sum()),
                    "rate_%": round(100 * flags.mean(), 4),
                    "eval_rows": len(flags),
                }
            )
    return rows


pg_summary = pd.DataFrame([r for pid in panel_data for r in per_group_rows(pid)])
if pg_summary.empty:
    print(
        "No channel groups configured (detect.groups is unset), so scoring ran as a single "
        "'system' scorer with no per-group breakdown. Set detect.groups in your config to "
        "enable grouped scoring; until then the combined-scorer table above is the per-run view."
    )
else:
    pg_summary = pg_summary.set_index(["panel", "scope", "group"]).sort_index()
pg_summary


In [ ]:
# --- Per-group raw scores + thresholds + flags ------------------------------
# One interactive figure per panel; rows = groups, columns = scorer fit scope.
# Each subplot draws the group's raw anomaly score (group_<name>_raw), the
# train-fitted flag threshold (dashed red, group_<name>_threshold), and the eval
# timestamps that group flagged (group_<name>_flag). The threshold is fit on the
# scorer-fit window, which differs by scope, so the same group can flag
# differently under "unseen" vs "all". Drag any subplot to zoom; double-click to
# reset.
group_colors_plotly = {"unseen (honest)": "#1f77b4", "all (optimistic)": "#ff7f0e"}
scope_labels = list(GROUP_SCOPE)

for pid in panel_data:
    # Union of groups across scopes (normally identical; be robust to a scope
    # whose scoring fell back to "system" with no groups).
    names = sorted({n for res in GROUP_SCOPE.values() for n in group_names(res[pid][0])})
    if not names:
        print(f"Panel {pid}: no channel groups configured; skipping per-group plot")
        continue

    def _flag_count(label, name):
        scores_df, flags_df = GROUP_SCOPE[label][pid][0], GROUP_SCOPE[label][pid][1]
        col = f"group_{name}_flag"
        return int(flags_df[col].astype(bool).sum()) if col in flags_df else 0

    titles = [f"{name} — {label} ({_flag_count(label, name)} flag)" for name in names for label in scope_labels]
    fig = make_subplots(
        rows=len(names),
        cols=len(GROUP_SCOPE),
        subplot_titles=titles,
        shared_xaxes=True,
        horizontal_spacing=0.06,
        vertical_spacing=min(0.04, 0.6 / max(len(names), 1)),
    )
    for c, (label, res) in enumerate(GROUP_SCOPE.items(), start=1):
        scores_df, flags_df = res[pid][0], res[pid][1]
        for r, name in enumerate(names, start=1):
            raw_col, thr_col, flag_col = f"group_{name}_raw", f"group_{name}_threshold", f"group_{name}_flag"
            if raw_col not in scores_df:
                continue
            s = scores_df[raw_col]
            flagged = flags_df[flag_col].astype(bool) if flag_col in flags_df else pd.Series(False, index=s.index)
            fig.add_trace(
                go.Scatter(
                    x=s.index, y=s.values, mode="lines",
                    line=dict(color=group_colors_plotly[label], width=1.0),
                    showlegend=False,
                    hovertemplate="%{x}<br>score=%{y:.3f}<extra></extra>",
                ),
                row=r, col=c,
            )
            t = float(scores_df[thr_col].dropna().iloc[0]) if scores_df[thr_col].notna().any() else float("nan")
            if np.isfinite(t):
                fig.add_hline(y=t, line=dict(color="red", dash="dash", width=1), row=r, col=c)
            if flagged.any():
                fig.add_trace(
                    go.Scatter(
                        x=s.index[flagged], y=s[flagged], mode="markers",
                        marker=dict(color="red", size=6),
                        showlegend=False,
                        hovertemplate="%{x}<br>FLAG %{y:.3f}<extra></extra>",
                    ),
                    row=r, col=c,
                )
    fig.update_layout(
        height=max(200 * len(names), 300),
        width=580 * len(GROUP_SCOPE),
        title_text=f"Per-group raw anomaly score — Panel {pid}",
        template="plotly_white",
    )
    fig.show()


In [ ]:
# --- Anomaly score over the ENTIRE dataset (all panels) ----------------------
# Detection only scores the eval (hist_window) tail. Here we score *every* row to
# see the residual profile across train/val/test, for each panel. Predictions are
# one-step-ahead on observed lags (predictor.predict). We build the *configured*
# scorer via the real pipeline so this works for any scorer (NormScorer / KMeans /
# IsolationForest); trainable scorers are fit on the whole-dataset residuals here
# (i.e. the "all"-scope baseline), while NormScorer's fit is a no-op.
from spotanomaly2.domain.spotforecast_adapter import SpotforecastPredictor

det_cfg = copy.deepcopy(config)
det_cfg["detect"]["model_timestamp"] = TRAINED_TIMESTAMP  # use the model trained above
det = AnomalyDetector(det_cfg, logger)
predictor = SpotforecastPredictor(det_cfg, logger)
scorer_name = det_cfg["detect"]["scorer_name"]


def score_full_dataset(panel_id: str):
    """Return (anomaly_score Series over the whole panel, model_data)."""
    df_full = panel_data[panel_id]
    model_data = det.load_forecasting_model(panel_id)
    pred = predictor.predict(model_data, df_full)
    true = df_full.loc[pred.index, list(pred.columns)]
    # Drop NaN/Inf rows — trainable scorers (IsolationForest, KMeans) can't fit on
    # NaN, and the detector likewise excludes imputed/invalid rows.
    finite = np.isfinite(true.to_numpy()).all(axis=1) & np.isfinite(pred.to_numpy()).all(axis=1)
    true_c, pred_c = true[finite], pred[finite]
    scores_df, _ = det.build_anomaly_detector().fit_score_detect(true_c, pred_c, true_c, pred_c)
    return scores_df["anomaly_score"], model_data


panels = list(panel_data)
fig = make_subplots(
    rows=len(panels),
    cols=1,
    subplot_titles=[
        f"Panel {pid} — {scorer_name} anomaly score over the full dataset" for pid in panels
    ],
    vertical_spacing=min(0.08, 0.5 / max(len(panels), 1)),
)

boundaries = [
    ("train_end_timestamp", "green", "train end"),
    ("val_end_timestamp", "orange", "val end"),
    ("test_start_timestamp", "red", "test start"),
]

score_full = {}
for i, pid in enumerate(panels, start=1):
    s, model_data = score_full_dataset(pid)
    score_full[pid] = s
    print(
        f"Panel {pid} [{scorer_name}]: scored {len(s):,} rows "
        f"({s.index.min()} -> {s.index.max()}); median={s.median():.3f}, max={s.max():.3f}"
    )
    fig.add_trace(
        go.Scatter(
            x=s.index, y=s.values, mode="lines",
            line=dict(color="#9467bd", width=0.8),
            name="anomaly_score", showlegend=(i == 1),
            hovertemplate="%{x}<br>score=%{y:.3f}<extra></extra>",
        ),
        row=i, col=1,
    )
    # Train/val/test boundaries persisted with the model (forecaster trained only
    # on the train split, so residuals there are in-sample/optimistically small).
    for key, color, lbl in boundaries:
        ts = model_data.get(key)
        if ts:
            x = pd.Timestamp(ts)
            fig.add_shape(
                type="line", x0=x, x1=x, yref="y domain", y0=0, y1=1,
                line=dict(color=color, dash="dash", width=1), row=i, col=1,
            )
            if i == 1:
                fig.add_annotation(
                    x=x, yref="y domain", y=1.0, text=lbl, showarrow=False,
                    yanchor="bottom", font=dict(size=9, color=color), row=i, col=1,
                )
    fig.update_yaxes(title_text="anomaly_score", row=i, col=1)

fig.update_xaxes(title_text="timestamp", row=len(panels), col=1)
fig.update_layout(
    height=380 * len(panels),
    width=1100,
    title_text="Full-dataset anomaly score (drag to zoom, double-click to reset)",
    template="plotly_white",
)
fig.show()


### Per-group scores over the full dataset

Same idea as the combined full-dataset score above, but broken out per channel
group. For each panel we predict one step ahead on observed lags across **every**
row, then score each configured group's channels with the detector's own
`_score_one` routine with the fit window set to the whole series — so each
group's threshold is fit on all rows (the optimistic in-sample baseline) and
every row is scored.

This is the wide-angle view: it shows each group's residual/score profile across
train/val/test (boundaries marked) so you can spot which group is driving flags
and where the residuals stop being optimistically small. Because the threshold is
fit in-sample over the whole series, flag counts here are *not* a production
estimate — use the leakage-guarded eval-window plots above for that.

> If no groups are configured for a panel, this cell prints a message and skips
> the panel (there is no per-group breakdown to plot).</cell id="4ad18f55">

In [ ]:
# --- Per-group score over the ENTIRE dataset --------------------------------
# Per-group counterpart to the combined full-dataset plot above. We resolve the
# configured groups and score each one with the detector's own _score_one
# routine, fit == eval == whole series -> thresholds fit on all rows (the
# "all"/optimistic baseline, matching the combined full-dataset cell) and every
# row is scored. Train/val/test boundaries are drawn so you can see where
# residuals turn from in-sample (optimistically small) to genuinely
# out-of-sample. Drag to zoom; x-axes are linked across groups (shared_xaxes).
#
# Reuses `det` / `predictor` built in the cell above.
def per_group_full(panel_id: str):
    """Return ({name: (score, threshold, flags)}, model_data) for the whole panel.

    Empty dict when no groups are configured for the panel.
    """
    df_full = panel_data[panel_id]
    model_data = det.load_forecasting_model(panel_id)
    pred = predictor.predict(model_data, df_full)
    true = df_full.loc[pred.index, list(pred.columns)]
    finite = np.isfinite(true.to_numpy()).all(axis=1) & np.isfinite(pred.to_numpy()).all(axis=1)
    true_c, pred_c = true[finite], pred[finite]
    groups = det._resolve_groups(panel_id, list(true_c.columns))
    out: dict[str, tuple] = {}
    if not groups:
        return out, model_data
    high_q = det.config["detect"]["high_quantile"]
    for name, g in groups.items():
        cols = g["columns"]
        # fit == eval == whole series for this group's channels.
        sc, fl = det._score_one(panel_id, f"group '{name}'", true_c[cols], pred_c[cols], true_c[cols], pred_c[cols], high_q)
        out[name] = (sc["anomaly_score"], sc["anomaly_threshold"], fl["anomaly_flag"].astype(bool))
    return out, model_data


boundaries = [
    ("train_end_timestamp", "green", "train end"),
    ("val_end_timestamp", "orange", "val end"),
    ("test_start_timestamp", "red", "test start"),
]

for pid in panel_data:
    groups_full, model_data = per_group_full(pid)
    if not groups_full:
        print(f"Panel {pid}: no channel groups configured; skipping per-group full-dataset plot")
        continue
    names = list(groups_full)
    flagged_counts = {n: int(groups_full[n][2].sum()) for n in names if groups_full[n][2].sum()}
    n_rows = len(next(iter(groups_full.values()))[0])
    print(f"Panel {pid}: per-group over {n_rows:,} scored rows; flagged groups: {flagged_counts}")

    titles = [f"{name} ({int(groups_full[name][2].sum())} flag)" for name in names]
    fig = make_subplots(
        rows=len(names),
        cols=1,
        subplot_titles=titles,
        shared_xaxes=True,
        vertical_spacing=min(0.04, 0.5 / max(len(names), 1)),
    )
    for r, name in enumerate(names, start=1):
        s, thr, flagged = groups_full[name]
        fig.add_trace(
            go.Scatter(
                x=s.index, y=s.values, mode="lines",
                line=dict(color="#9467bd", width=0.8), showlegend=False,
                hovertemplate="%{x}<br>score=%{y:.3f}<extra></extra>",
            ),
            row=r, col=1,
        )
        t = float(thr.dropna().iloc[0]) if thr.notna().any() else float("nan")
        if np.isfinite(t):
            fig.add_hline(y=t, line=dict(color="red", dash="dash", width=1), row=r, col=1)
        if flagged.any():
            fig.add_trace(
                go.Scatter(
                    x=s.index[flagged], y=s[flagged], mode="markers",
                    marker=dict(color="red", size=5), showlegend=False,
                    hovertemplate="%{x}<br>FLAG %{y:.3f}<extra></extra>",
                ),
                row=r, col=1,
            )
        for key, color, lbl in boundaries:
            ts = model_data.get(key)
            if ts:
                x = pd.Timestamp(ts)
                fig.add_shape(
                    type="line", x0=x, x1=x, yref="y domain", y0=0, y1=1,
                    line=dict(color=color, dash="dot", width=1), row=r, col=1,
                )
                if r == 1:
                    fig.add_annotation(
                        x=x, yref="y domain", y=1.0, text=lbl, showarrow=False,
                        yanchor="bottom", font=dict(size=9, color=color), row=r, col=1,
                    )
    fig.update_xaxes(title_text="timestamp", row=len(names), col=1)
    fig.update_layout(
        height=max(180 * len(names), 300),
        width=1100,
        title_text=f"Per-group anomaly score over the full dataset — Panel {pid} (thresholds fit on all rows)",
        template="plotly_white",
    )
    fig.show()


## Takeaway

`"unseen"` is the safe default for production monitoring: the scorer's normal
baseline is calibrated only on data the forecaster and tuner never touched, so the
flag rate is honest.

`"all"` is useful when you genuinely have very little held-out data and need the
scorer to see more rows to estimate a stable distribution — accepting that the
baseline is optimistically biased. The `only_unseen` column above quantifies exactly
what you give up: anomalies that the honest calibration would have caught.

Flip the scope in `config/config.yaml`:

```yaml
detect:
  scorer_fit_scope: "all"   # default is "unseen"
```